# 🕵️ Dia 03 — TCP/IP para investigadores: lendo uma conexão

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 3* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-03.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

12h31. Você capturou o laboratório pedindo `serraazul-pix.example` em HTTP puro: DNS, aperto de mão, `GET`, `301`, dois `FIN`. Depois tentou `serraazul-seguranca.example` (`203.0.113.78`): quatro `SYN` e nada.

Rafa: *"Nada não é resultado. Nada tem formato. Me diz qual é."*

Neste notebook você vai:

1. **Ler um pcap só com `struct`**, entendendo cada campo de Ethernet, IP, UDP e TCP.
2. **Imprimir a lista de pacotes como o Wireshark**, em UTC, e gerar a escada em mermaid sozinha.
3. **Jogar o quiz "que formato é este?"** (6 perguntas).
4. **Gerar a tabela de quíntupla** do Passo 4 para a captura da Operação Isca.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

## 1. Carregar a captura

O arquivo `isca-lab.pcap` está no repositório. Se não houver internet, usamos a cópia embutida abaixo (o pcap inteiro em *base64*, que é só um jeito de escrever bytes como texto). Antes de qualquer análise: **hash**. Disciplina do Dia 2.

In [ ]:
import base64, hashlib, urllib.request

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-03/isca-lab.pcap"

# Plano B: o mesmo arquivo, em base64 (texto que representa bytes).
PCAP_B64 = (
    "1MOyoQIABAAAAAAAAAAAAP//AAABAAAAu8LDarqPAQBRAAAAUQAAAAoAJwAAAQgAJwoLDAgARQAA"
    "QwPpQABAEUVlwKg4CsCoOAGjlQA1AC9gpFoRAQAAAQAAAAAAAA1zZXJyYWF6dWwtcGl4B2V4YW1w"
    "bGUAAAEAAbvCw2qC2wEAYQAAAGEAAAAIACcKCwwKACcAAAEIAEUAAFMImUAAQBFApcCoOAHAqDgK"
    "ADWjlQA/UwVaEYGAAAEAAQAAAAANc2VycmFhenVsLXBpeAdleGFtcGxlAAABAAHADAABAAEAAAEs"
    "AATLAHFNu8LDasLhAQA6AAAAOgAAAAoAJwAAAQgAJwoLDAgARQAALAPqQABABgHiwKg4CssAcU3J"
    "QgBQteiiUQAAAABgAvrwRmgAAAIEBbS7wsNqkr4EADoAAAA6AAAACAAnCgsMCgAnAAABCABFAAAs"
    "AABAADEGFMzLAHFNwKg4CgBQyUJpkl9ZteiiUmAS//94XAAAAgQFtLvCw2pavwQANgAAADYAAAAK"
    "ACcAAAEIACcKCwwIAEUAACgD60AAQAYB5cCoOArLAHFNyUIAULXoolJpkl9aUBD68JUoAAC7wsNq"
    "hsAEAIoAAACKAAAACgAnAAABCAAnCgsMCABFAAB8A+xAAEAGAZDAqDgKywBxTclCAFC16KJSaZJf"
    "WlAY+vC/PwAAR0VUIC8gSFRUUC8xLjENCkhvc3Q6IHNlcnJhYXp1bC1waXguZXhhbXBsZQ0KVXNl"
    "ci1BZ2VudDogY3VybC84LjUuMA0KQWNjZXB0OiAqLyoNCg0Ku8LDapqaBwA2AAAANgAAAAgAJwoL"
    "DAoAJwAAAQgARQAAKKgNQAAxBmzCywBxTcCoOAoAUMlCaZJfWrXooqZQEP//j8UAALvCw2oqnAcA"
    "mQEAAJkBAAAIACcKCwwKACcAAAEIAEUAAYuoDkAAMQZrXssAcU3AqDgKAFDJQmmSX1q16KKmUBj/"
    "/9IVAABIVFRQLzEuMSAzMDEgTW92ZWQgUGVybWFuZW50bHkNClNlcnZlcjogbmdpbngNCkRhdGU6"
    "IE1vbiwgMDUgT2N0IDIwMjYgMTU6MzE6MDcgR01UDQpDb250ZW50LVR5cGU6IHRleHQvaHRtbA0K"
    "Q29udGVudC1MZW5ndGg6IDE2Mg0KTG9jYXRpb246IGh0dHBzOi8vc2VycmFhenVsLXBpeC5leGFt"
    "cGxlLw0KQ29ubmVjdGlvbjogY2xvc2UNCg0KPGh0bWw+DQo8aGVhZD48dGl0bGU+MzAxIE1vdmVk"
    "IFBlcm1hbmVudGx5PC90aXRsZT48L2hlYWQ+DQo8Ym9keT4NCjxjZW50ZXI+PGgxPjMwMSBNb3Zl"
    "ZCBQZXJtYW5lbnRseTwvaDE+PC9jZW50ZXI+DQo8aHI+PGNlbnRlcj5uZ2lueDwvY2VudGVyPg0K"
    "PC9ib2R5Pg0KPC9odG1sPg0Ku8LDah6eBwA2AAAANgAAAAoAJwAAAQgAJwoLDAgARQAAKAPtQABA"
    "BgHjwKg4CssAcU3JQgBQteiipmmSYL1QEPrwk3EAALvCw2qunwcANgAAADYAAAAIACcKCwwKACcA"
    "AAEIAEUAACioD0AAMQZswMsAcU3AqDgKAFDJQmmSYL216KKmUBH//45hAAC7wsNqBqIHADYAAAA2"
    "AAAACgAnAAABCAAnCgsMCABFAAAoA+5AAEAGAeLAqDgKywBxTclCAFC16KKmaZJgvlAR+vCTbwAA"
    "u8LDasJ5CgA2AAAANgAAAAgAJwoLDAoAJwAAAQgARQAAKKgQQAAxBmy/ywBxTcCoOAoAUMlCaZJg"
    "vrXooqdQEP//jmAAAOTCw2r6nAQAVwAAAFcAAAAKACcAAAEIACcKCwwIAEUAAEkD8kAAQBFFVsCo"
    "OArAqDgBo60ANQA1JEtaEgEAAAEAAAAAAAATc2VycmFhenVsLXNlZ3VyYW5jYQdleGFtcGxlAAAB"
    "AAHkwsNqvvEEAGcAAABnAAAACAAnCgsMCgAnAAABCABFAABZCJpAAEARQJ7AqDgBwKg4CgA1o60A"
    "RRWsWhKBgAABAAEAAAAAE3NlcnJhYXp1bC1zZWd1cmFuY2EHZXhhbXBsZQAAAQABwAwAAQABAAAB"
    "LAAEywBxTuTCw2rS9gQAOgAAADoAAAAKACcAAAEIACcKCwwIAEUAACwD80AAQAYB2MCoOArLAHFO"
    "yUoAUIPJAXAAAAAAYAL68BlgAAACBAW05cLDaoZWBQA6AAAAOgAAAAoAJwAAAQgAJwoLDAgARQAA"
    "LAP0QABABgHXwKg4CssAcU7JSgBQg8kBcAAAAABgAvrwGWAAAAIEBbTnwsNqlpYFADoAAAA6AAAA"
    "CgAnAAABCAAnCgsMCABFAAAsA/VAAEAGAdbAqDgKywBxTslKAFCDyQFwAAAAAGAC+vAZYAAAAgQF"
    "tOvCw2rahwcAOgAAADoAAAAKACcAAAEIACcKCwwIAEUAACwD9kAAQAYB1cCoOArLAHFOyUoAUIPJ"
    "AXAAAAAAYAL68BlgAAACBAW0"
)
SHA_ESPERADO = "e040a0433e53fb4e0adefc62bd4b203a2dec6cc66743a9965a1c01d10cc02766"

try:
    with urllib.request.urlopen(URL, timeout=5) as r:      # tenta o GitHub
        dados = r.read()
    if hashlib.sha256(dados).hexdigest() != SHA_ESPERADO:   # se o arquivo do repo mudou...
        raise ValueError("hash diferente do esperado")      # ...preferimos a cópia conhecida
    print("🌐 baixado do GitHub")
except Exception as erro:
    dados = base64.b64decode(PCAP_B64)                      # volta a ser bytes
    print(f"📦 usando a cópia embutida ({type(erro).__name__})")

print("bytes:", len(dados))
print("SHA-256:", hashlib.sha256(dados).hexdigest())
print("confere com o esperado?", hashlib.sha256(dados).hexdigest() == SHA_ESPERADO)

## 2. Um leitor de pcap em 40 linhas

O pcap clássico tem:

| Parte | Tamanho | O que guarda |
|---|---|---|
| Cabeçalho global | 24 bytes | número mágico `a1b2c3d4`, versão, tipo de enlace (1 = Ethernet) |
| Cabeçalho de cada pacote | 16 bytes | segundos, microssegundos (UTC!), tamanho gravado, tamanho original |
| Pacote | variável | Ethernet (14 bytes) → IPv4 (20+) → TCP (20+) ou UDP (8) → dados |

`struct.unpack` transforma bytes em números segundo um "molde": `<` = little-endian (como o pcap grava), `!` = ordem de rede (como o IP grava), `I` = 4 bytes, `H` = 2 bytes, `B` = 1 byte.

In [ ]:
import struct
from datetime import datetime, timezone

FLAGS = [(0x02, "SYN"), (0x10, "ACK"), (0x01, "FIN"), (0x04, "RST"), (0x08, "PSH")]   # bit → nome

def ler_pcap(dados):
    """Devolve uma lista de dicionários, um por pacote, com os campos que o investigador usa."""
    magico = struct.unpack("<I", dados[:4])[0]
    assert magico == 0xA1B2C3D4, "não é pcap clássico little-endian (pcapng? converta com editcap)"
    pos, n, pacotes = 24, 0, []                          # pula o cabeçalho global
    while pos < len(dados):
        seg, us, grav, orig = struct.unpack("<IIII", dados[pos:pos + 16])   # cabeçalho do pacote
        q = dados[pos + 16: pos + 16 + grav]             # q = o quadro Ethernet inteiro
        pos += 16 + grav                                 # avança para o próximo pacote
        n += 1
        p = {"n": n, "hora": datetime.fromtimestamp(seg + us / 1e6, timezone.utc), "tam": orig}
        if struct.unpack("!H", q[12:14])[0] != 0x0800:   # EtherType 0x0800 = IPv4
            pacotes.append(p); continue
        ip = q[14:]                                      # começa o cabeçalho IP
        ihl = (ip[0] & 0x0F) * 4                         # tamanho do cabeçalho IP, em bytes
        p["ttl"], p["proto"] = ip[8], ip[9]              # TTL e protocolo (6 = TCP, 17 = UDP)
        p["src"] = ".".join(map(str, ip[12:16]))         # IP de origem
        p["dst"] = ".".join(map(str, ip[16:20]))         # IP de destino
        l4 = ip[ihl:]                                    # camada 4 (TCP ou UDP)
        p["sport"], p["dport"] = struct.unpack("!HH", l4[:4])
        if p["proto"] == 6:                              # TCP
            p["seq"], p["ack"] = struct.unpack("!II", l4[4:12])
            off = (l4[12] >> 4) * 4                      # tamanho do cabeçalho TCP
            fl = l4[13]                                  # o byte das flags
            p["flags"] = [nome for bit, nome in FLAGS if fl & bit]
            p["dados"] = l4[off:]                        # payload (ex.: o texto do HTTP)
        else:                                            # UDP: 8 bytes de cabeçalho
            p["flags"], p["dados"] = [], l4[8:]
        pacotes.append(p)
    return pacotes

PACOTES = ler_pcap(dados)
print(len(PACOTES), "pacotes lidos")

## 3. A lista de pacotes, como no Wireshark

Números de sequência **relativos** (começando em 0), como o Wireshark mostra por padrão. Para isso guardamos o primeiro `seq` de cada lado da conversa.

In [ ]:
inicio_seq = {}   # (ip, porta) → primeiro número de sequência visto daquele lado

def info(p):
    """Monta a coluna 'Info' parecida com a do Wireshark."""
    if p["proto"] == 17:
        return f"DNS {p['sport']} → {p['dport']}"
    corpo = p["dados"]
    if corpo[:4] in (b"GET ", b"POST") or corpo[:5] == b"HTTP/":
        return corpo.split(b"\r\n")[0].decode()       # primeira linha do HTTP
    lado = (p["src"], p["sport"]); outro = (p["dst"], p["dport"])
    inicio_seq.setdefault(lado, p["seq"])               # primeira vez que vemos este lado
    rs = p["seq"] - inicio_seq[lado]
    ra = p["ack"] - inicio_seq.get(outro, p["ack"]) if "ACK" in p["flags"] else 0
    return f"{p['sport']} → {p['dport']} [{', '.join(p['flags'])}] Seq={rs} Ack={ra}"

print(f"{'No.':>3}  {'Hora (UTC)':26}  {'Origem':15} {'Destino':15} {'Proto':5} TTL  Info")
for p in PACOTES:
    proto = {6: "TCP", 17: "UDP"}.get(p["proto"], "?")
    if p["proto"] == 6 and p["dados"][:5] in (b"GET /", b"HTTP/"):
        proto = "HTTP"
    print(f"{p['n']:>3}  {p['hora'].strftime('%Y-%m-%d %H:%M:%S.%f'):26}  "
          f"{p['src']:15} {p['dst']:15} {proto:5} {p['ttl']:>3}  {info(p)}")

### 🔎 Os filtros do Passo 2, em Python

Cada filtro de exibição do Wireshark vira uma condição. Mude `FILTRO` e rode de novo.

In [ ]:
#@title Filtro de exibição { display-mode: "form" }
FILTRO = "so_aberturas"  #@param ["dns", "so_aberturas", "conversa_77", "http_request", "fim_ou_reset", "tentativa_78"]

FILTROS = {
    "dns":          lambda p: p["proto"] == 17 and 53 in (p["sport"], p["dport"]),            # dns
    "so_aberturas": lambda p: "SYN" in p["flags"] and "ACK" not in p["flags"],                # tcp.flags.syn==1 && tcp.flags.ack==0
    "conversa_77":  lambda p: "203.0.113.77" in (p["src"], p["dst"]) and 80 in (p["sport"], p["dport"]),
    "http_request": lambda p: p["proto"] == 6 and p["dados"][:4] == b"GET ",                   # http.request
    "fim_ou_reset": lambda p: "FIN" in p["flags"] or "RST" in p["flags"],                     # tcp.flags.fin==1 || tcp.flags.reset==1
    "tentativa_78": lambda p: "203.0.113.78" in (p["src"], p["dst"]),                          # ip.addr == 203.0.113.78
}
for p in filter(FILTROS[FILTRO], PACOTES):
    print(p["n"], p["hora"].strftime("%H:%M:%S.%f"), p["src"], "→", p["dst"], p["flags"] or "UDP")

## 4. A escada (flow graph) gerada sozinha

O Passo 3 pede o diagrama em escada. Aqui o código escreve o mermaid; cole num bloco ` ```mermaid ` em `day03-flow-isca.md` e o GitHub desenha.

In [ ]:
conv = [p for p in PACOTES if FILTROS["conversa_77"](p)]
cli = f"{conv[0]['src']} porta {conv[0]['sport']} (cliente)"     # quem mandou o 1º SYN é o cliente
srv = f"{conv[0]['dst']} porta {conv[0]['dport']} (servidor)"
t0 = conv[0]["hora"]

linhas = ["sequenceDiagram", f"    participant C as {cli}", f"    participant S as {srv}"]
for p in conv:
    seta = "C->>S" if p["src"] == conv[0]["src"] else "S->>C"     # direção da seta
    ms = (p["hora"] - t0).total_seconds() * 1000                   # tempo desde o 1º SYN
    linhas.append(f"    {seta}: pct {p['n']} {info(p)} (+{ms:.1f} ms)")
ESCADA = "\n".join(linhas)
print(ESCADA)

rtt = (conv[1]["hora"] - conv[0]["hora"]).total_seconds() * 1000
print(f"\nRTT (SYN → SYN-ACK): {rtt:.1f} ms")

## 5. TTL e a tentativa que falhou

In [ ]:
def ttl_inicial(ttl):
    """Menor valor 'redondo' acima do observado. É INFERÊNCIA: o valor inicial pode ser alterado."""
    return next(v for v in (32, 64, 128, 255) if ttl <= v)

ttl_srv = conv[1]["ttl"]                                        # o SYN-ACK veio do servidor
print(f"TTL observado do servidor: {ttl_srv} → inicial provável {ttl_inicial(ttl_srv)}, "
      f"~{ttl_inicial(ttl_srv) - ttl_srv} saltos (inferência)")

syns = [p for p in PACOTES if FILTROS["tentativa_78"](p) and p["proto"] == 6]
print(f"\nTentativa com 203.0.113.78: {len(syns)} SYNs, nenhuma resposta")
for a, b in zip(syns, syns[1:]):
    print(f"  intervalo: {(b['hora'] - a['hora']).total_seconds():.2f} s")   # o cliente dobra a espera
print("Formato: SYN sem resposta + retransmissões → filtrado, fora do ar OU inexistente (3 hipóteses).")

## 6. 🎮 Quiz: que formato é este?

Responda com o número da opção. `MODO_JOGO = False` mostra só as respostas.

In [ ]:
OPCOES = ["1 = aperto de mão completo", "2 = recusado (SYN → RST)",
          "3 = sem resposta (filtrado / fora / inexistente)", "4 = varredura de portas"]
PERGUNTAS = [
    ("SYN, SYN-ACK, ACK, dados, FIN, FIN, ACK entre a VM e 203.0.113.77:80", "1",
     "É a conversa completa da captura: abriu, trocou dados e fechou em ordem."),
    ("Quatro SYNs para 203.0.113.78:80, intervalos de ~1, 2 e 4 s, nada volta", "3",
     "Retransmissões com espera dobrando são a assinatura de SYN sem resposta."),
    ("Um SYN para a porta 81 e, 20 ms depois, um RST vindo do mesmo IP", "2",
     "O host está no ar e respondeu; só não há nada escutando na porta."),
    ("De um único IP externo: SYN para as portas 21, 22, 23, 25, 80, 443, 3389... nenhum aperto completo", "4",
     "Muitas portas, nenhuma conversa: varredura."),
    ("Log de firewall: 198.51.100.14 → 10.20.32.10 TCP/3389 BLOCK, um só registro", "3",
     "Para quem tentou, um bloqueio silencioso é igual a 'sem resposta'. (Um registro só não é varredura.)"),
    ("SYN, SYN-ACK, ACK, depois RST do cliente sem FIN nenhum", "1",
     "Pegadinha: a conexão ABRIU (aperto completo); só terminou de forma abrupta. É o RSTO do Zeek, amanhã."),
]

MODO_JOGO = True

acertos = 0
print("\n".join(OPCOES))
for i, (cena, certo, porque) in enumerate(PERGUNTAS, start=1):
    print(f"\n[{i}/{len(PERGUNTAS)}] {cena}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3", "4"):
            r = input("   Formato (1-4): ").strip()
        acertos += (r == certo)
        print("   ✅ Acertou!" if r == certo else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {OPCOES[int(certo) - 1]} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(PERGUNTAS)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 5 else "— meta: 5 de 6.")

## 7. 📝 A tabela de quíntupla (Passo 4) para a Operação Isca

A tabela sai preenchida a partir dos pacotes. Confira cada célula com a lista do item 3: o código pode errar, você assina.

In [ ]:
from pathlib import Path

ida = [p for p in conv if p["src"] == conv[0]["src"]]           # cliente → servidor
volta = [p for p in conv if p["src"] != conv[0]["src"]]         # servidor → cliente
hs = [p["n"] for p in conv[:3]]                                 # os 3 do aperto de mão
primeiro_fin = next(p for p in conv if "FIN" in p["flags"])
quem_fecha = "servidor" if primeiro_fin["src"] == conv[0]["dst"] else "cliente"

tabela = f"""| Campo | Conversa com 203.0.113.77 | Tentativa com 203.0.113.78 |
|---|---|---|
| Protocolo | TCP | TCP |
| IP : porta do cliente | {conv[0]['src']}:{conv[0]['sport']} | {syns[0]['src']}:{syns[0]['sport']} |
| IP : porta do servidor | {conv[0]['dst']}:{conv[0]['dport']} | {syns[0]['dst']}:{syns[0]['dport']} |
| Primeiro pacote (UTC) | {conv[0]['hora'].isoformat()} | {syns[0]['hora'].isoformat()} |
| Duração (s) | {(conv[-1]['hora'] - conv[0]['hora']).total_seconds():.3f} | {(syns[-1]['hora'] - syns[0]['hora']).total_seconds():.3f} |
| Pacotes A→B / B→A | {len(ida)} / {len(volta)} | {len(syns)} / 0 |
| Bytes A→B / B→A | {sum(p['tam'] for p in ida)} / {sum(p['tam'] for p in volta)} | {sum(p['tam'] for p in syns)} / 0 |
| Como abriu | SYN / SYN-ACK / ACK nos pacotes {hs[0]}, {hs[1]}, {hs[2]} | não abriu: {len(syns)} SYNs sem resposta |
| Como fechou | FIN do {quem_fecha} primeiro (pacote {primeiro_fin['n']}) | — |
| TTL observado do servidor / inicial provável | {ttl_srv} / {ttl_inicial(ttl_srv)} (inferência) | — |
| SHA-256 do arquivo de captura | {hashlib.sha256(dados).hexdigest()} | idem |
"""
Path("notas").mkdir(exist_ok=True)
Path("notas/day03-5tuple-isca.md").write_text(tabela + "\n```mermaid\n" + ESCADA + "\n```\n", encoding="utf-8")
print(tabela)
print("💾 Salvo em notas/day03-5tuple-isca.md (com a escada mermaid no fim)")
try:
    from google.colab import files
    files.download("notas/day03-5tuple-isca.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou na pasta notas/)")

## ✅ Antes de fechar o Dia 3

- [ ] Hash do `isca-lab.pcap` anotado no registro de custódia
- [ ] Sabe dizer quem fechou a conexão primeiro e qual foi a resposta HTTP
- [ ] Quiz com pelo menos 5/6
- [ ] `notas/day03-5tuple-isca.md` gerado e conferido
- [ ] Fez também a prática original no Wireshark (`neverssl.com`, porta 81)

**Amanhã (Dia 4):** a cooperativa entrega o `conn.log` da rede da sede. Uma linha por conexão, nenhum pacote. Quem, de dentro da Serra Azul, falou com `203.0.113.77`?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>